# Predictive Analysis of Sentiment Percentage

In this dataset, it we have question that have different options that contribute to Positive, nuetral and negative. Below is dividied into 3 sections of each kind of predictions based on the question response type for categories over the years.

## All questions that contribute to Sentiments
Dataset Includes all questions that have response type that contributes to positive, negative and may have neutral.

In [50]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Path to the folder containing all CSV files
folder_path = "../data/clean/"

# Combine all CSV files into a single dataframe
all_dataframes = []
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        temp_df = pd.read_csv(file_path)
        all_dataframes.append(temp_df)

# Merge all dataframes
df = pd.concat(all_dataframes, ignore_index=True)

target_cols = ["MOST_POSITIVE_OR_LEAST_NEGATIVE", "NEUTRAL_OR_MIDDLE_CATEGORY", "MOST_NEGATIVE_OR_LEAST_POSITIVE"]
for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Ensure ANSWER1 and ANSWER2 columns exist and convert them to numeric
if "ANSWER1" in df.columns:
    df["ANSWER1"] = pd.to_numeric(df["ANSWER1"], errors='coerce')
if "ANSWER2" in df.columns:
    df["ANSWER2"] = pd.to_numeric(df["ANSWER2"], errors='coerce')

# Update values based on QUESTION conditions
df.loc[df["QUESTION"].isin(["Q57", "Q64"]), "MOST_POSITIVE_OR_LEAST_NEGATIVE"] = df["ANSWER2"]
df.loc[df["QUESTION"].isin(["Q57", "Q64"]), "MOST_NEGATIVE_OR_LEAST_POSITIVE"] = df["ANSWER1"]
df.loc[df["QUESTION"].isin(["Q57", "Q64"]), "NEUTRAL_OR_MIDDLE_CATEGORY"] = 0

df.loc[df["QUESTION"].isin(["Q85", "Q89"]), "MOST_POSITIVE_OR_LEAST_NEGATIVE"] = df["ANSWER1"]
df.loc[df["QUESTION"].isin(["Q85", "Q89"]), "MOST_NEGATIVE_OR_LEAST_POSITIVE"] = df["ANSWER2"]
df.loc[df["QUESTION"].isin(["Q85", "Q89"]), "NEUTRAL_OR_MIDDLE_CATEGORY"] = 0

# Drop rows where all three target columns have 9999 values
# Mostly questions - Q56_1,Q56_2,Q58,Q59,Q60,Q61,Q65,Q66,Q67,Q68,Q83,Q84,Q88,Q92,Q94,Q95,Q96
df = df[~((df["MOST_POSITIVE_OR_LEAST_NEGATIVE"] == 9999) & 
          (df["MOST_NEGATIVE_OR_LEAST_POSITIVE"] == 9999) & 
          (df["NEUTRAL_OR_MIDDLE_CATEGORY"] == 9999))]

# Selecting relevant columns
features = ["BYCOND", "descrip_E", "SURVEYR", "QUESTION", "INDICATORENG", "SUBINDICATORENG"]
targets = ["MOST_POSITIVE_OR_LEAST_NEGATIVE", "NEUTRAL_OR_MIDDLE_CATEGORY", "MOST_NEGATIVE_OR_LEAST_POSITIVE"]

df = df[features + targets]

# Ensure SURVEYR (year) is numeric
df["SURVEYR"] = pd.to_numeric(df["SURVEYR"], errors="coerce")
df.dropna(subset=["SURVEYR"], inplace=True)

# Encoding categorical features
label_encoders = {}
categorical_cols = ["BYCOND", "descrip_E", "QUESTION", "INDICATORENG", "SUBINDICATORENG"]
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Ensure target columns are numeric
for target in targets:
    df[target] = pd.to_numeric(df[target], errors="coerce")
df.dropna(subset=targets, inplace=True)

# Splitting data
X = df[features]
y = df[targets]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Store original test values before transformation
X_test_original = X_test.copy()

# Scaling numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Training XGBoost models
models = {}
for i, sentiment in enumerate(targets):
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train.iloc[:, i])
    models[sentiment] = model

# Predictions
y_pred = np.column_stack([models[sent].predict(X_test) for sent in targets])

# Evaluation
for i, sentiment in enumerate(targets):
    print(f"{sentiment} - MAE: {mean_absolute_error(y_test.iloc[:, i], y_pred[:, i]):.2f}")

# Function to predict sentiment percentages
def predict_sentiment(new_data):
    new_data_df = pd.DataFrame([new_data], columns=features)
    
    for col in categorical_cols:
        if new_data_df[col][0] not in label_encoders[col].classes_:
            label_encoders[col].classes_ = np.append(label_encoders[col].classes_, new_data_df[col][0])
        
        try:
            new_data_df[col] = label_encoders[col].transform(new_data_df[col].astype(str))
        except ValueError:
            print(f"Warning: Unable to transform column '{col}' with value '{new_data_df[col][0]}'. Setting default value of -1.")
            new_data_df[col] = -1  
    
    new_data_df["SURVEYR"] = pd.to_numeric(new_data_df["SURVEYR"], errors="coerce")
    new_data_df.fillna(0, inplace=True)
    
    new_data_df = scaler.transform(new_data_df)
    predictions = {sent: models[sent].predict(new_data_df)[0] for sent in targets}
    return predictions


MOST_POSITIVE_OR_LEAST_NEGATIVE - MAE: 6.09
NEUTRAL_OR_MIDDLE_CATEGORY - MAE: 4.06
MOST_NEGATIVE_OR_LEAST_POSITIVE - MAE: 4.53


In [51]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "",
        "descrip_E": "Health Canada",
        "SURVEYR": "2020",
        "QUESTION":"Q80",
        "TITLE_E": "Question 80. I would describe my workplace as being psychologically healthy.",
        "INDICATORENG": "Workplace well-being","SUBINDICATORENG": "A psychologically healthy workplace"
    },
    {
        "BYCOND": "Q93 = 2",
        "descrip_E": "Working remotely",
        "SURVEYR": "2022",
        "QUESTION":"Q72g",
        "TITLE_E": "Question 72g. Overall, to what extent do the following factors cause you work-related stress? Lack of control or input in decision-making",
        "INDICATORENG": "Workplace well-being","SUBINDICATORENG": "Work-related stress"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 70.83
Neutral or Middle Category: 13.93
Most Negative or Least Positive: 16.11

Sample 2 Prediction:
Most Positive or Least Negative: 61.84
Neutral or Middle Category: 19.85
Most Negative or Least Positive: 17.65



In [54]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "Q114 = 7",
        "descrip_E": "British Columbia",
        "SURVEYR": "2019",
        "QUESTION":"Q21",
        "TITLE_E": "Question 21. In my work unit, individuals behave in a respectful manner.",
        "INDICATORENG": "Workplace","SUBINDICATORENG": "Diversity and inclusion"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 84.26
Neutral or Middle Category: 7.47
Most Negative or Least Positive: 8.45



In [56]:
# Sample Inputs for Prediction
sample_inputs = [
    {
        "BYCOND": "Q114 = 7",
        "descrip_E": "British Columbia",
        "SURVEYR": "2023",
        "QUESTION":"Q21",
        "TITLE_E": "Question 21. In my work unit, individuals behave in a respectful manner.",
        "INDICATORENG": "Workplace","SUBINDICATORENG": "Diversity and inclusion"
    }
]

for i, sample in enumerate(sample_inputs):
    prediction = predict_sentiment(sample)  # This function applies your trained models
    print(f"Sample {i+1} Prediction:")
    print(f"Most Positive or Least Negative: {prediction['MOST_POSITIVE_OR_LEAST_NEGATIVE']:.2f}")
    print(f"Neutral or Middle Category: {prediction['NEUTRAL_OR_MIDDLE_CATEGORY']:.2f}")
    print(f"Most Negative or Least Positive: {prediction['MOST_NEGATIVE_OR_LEAST_POSITIVE']:.2f}\n")


Sample 1 Prediction:
Most Positive or Least Negative: 87.51
Neutral or Middle Category: 6.79
Most Negative or Least Positive: 6.72

